In [1]:
from a01_clean_data import clean_data
from b02_fetch_weather import fetch_weather, clean_weather_data
import pandas as pd
import matplotlib as mpl
import openmeteo_requests
import requests_cache
from retry_requests import retry
import holidays
ca_holidays = holidays.US(subdiv='CA', years=[2024, 2025, 2026])

dfgym, mindate, maxdate = clean_data('../data/raw/data.csv')
dfweather = fetch_weather(mindate, maxdate)
dfweather = clean_weather_data(dfweather)
dfweather.to_csv("../data/processed/clean_weather_data.csv", index=False) # Export as a csv

In [2]:
# Merge dataframes
df = pd.merge(dfgym, dfweather, on=["key_date", "hour"], how="left")

In [3]:
# Reorder columns
df = df[["key_date","year","month","day","hour","minute","number_people","day_of_week","is_weekend","is_holiday","is_start_of_semester","is_during_semester","temp_f","rain_mm","encoded_weather_code","weather_type"]]
df[df["day_of_week"] == "Friday"]

,key_date,year,month,day,hour,minute,number_people,day_of_week,is_weekend,is_holiday,is_start_of_semester,is_during_semester,temp_f,rain_mm,encoded_weather_code,weather_type
0,2015-08-14,2015,8,14,17,0,37,Friday,False,False,False,False,80.6576,0.0,0,Clear sky
1,2015-08-14,2015,8,14,17,20,45,Friday,False,False,False,False,80.6576,0.0,0,Clear sky
2,2015-08-14,2015,8,14,17,30,40,Friday,False,False,False,False,80.6576,0.0,0,Clear sky
3,2015-08-14,2015,8,14,17,40,44,Friday,False,False,False,False,80.6576,0.0,0,Clear sky
4,2015-08-14,2015,8,14,17,50,45,Friday,False,False,False,False,80.6576,0.0,0,Clear sky
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62075,2017-03-17,2017,3,17,23,20,7,Friday,False,False,False,True,54.2300,0.0,3,Overcast
62076,2017-03-17,2017,3,17,23,30,5,Friday,False,False,False,True,54.2300,0.0,3,Overcast
62077,2017-03-17,2017,3,17,23,40,2,Friday,False,False,False,True,54.2300,0.0,3,Overcast
62078,2017-03-17,2017,3,17,23,50,0,Friday,False,False,False,True,54.2300,0.0,3,Overcast


In [4]:
# Export as a csv for further use
df.to_csv("../data/processed/clean_gym_data.csv", index=False)

### Analysis

In [5]:
clear_weather = df[df["rain_mm"] == 0]
clear_weather_avg = clear_weather.groupby("day_of_week")["number_people"].mean()

rainy_weather = df[df["rain_mm"] > 0]
rainy_weather_avg = rainy_weather.groupby("day_of_week")["number_people"].mean()

merged_df_2 = pd.merge(clear_weather_avg, rainy_weather_avg, on="day_of_week").reset_index()
merged_df_2.rename(columns={"number_people_x":"clear_avg","number_people_y":"rainy_avg"}, inplace=True)
merged_df_2["rain_penalty"] = merged_df_2["rainy_avg"] - merged_df_2["clear_avg"]
rain_penalty = merged_df_2

rain_penalty = rain_penalty[["day_of_week","rain_penalty"]]
rain_penalty

,day_of_week,rain_penalty
0,Friday,3.937187
1,Monday,-9.977016
2,Saturday,1.645923
3,Sunday,2.034177
4,Thursday,1.737461
5,Tuesday,-12.127372
6,Wednesday,-2.294729


In [6]:
is_holiday = df
is_holiday = is_holiday.groupby("is_holiday")["number_people"].mean().reset_index()

no_holiday_avg = is_holiday[is_holiday["is_holiday"] == False]["number_people"].values[0]
holiday_avg = is_holiday[is_holiday["is_holiday"] == True]["number_people"].values[0]

holiday_penalty = holiday_avg - no_holiday_avg

In [ ]:
base = clear_weather
base = base.groupby(["day_of_week","hour"])["number_people"].mean().reset_index()
# base

keys = list(zip(base["day_of_week"], base["hour"]))
values = base["number_people"]
reference_base = dict(zip(keys,values))
# reference_base

{('Friday', 0): 21.308108108108108,
 ('Friday', 1): 3.6869565217391305,
 ('Friday', 2): 0.1836734693877551,
 ('Friday', 3): 0.21875,
 ('Friday', 4): 0.34806629834254144,
 ('Friday', 5): 1.3453947368421053,
 ('Friday', 6): 10.819718309859155,
 ('Friday', 7): 19.124293785310734,
 ('Friday', 8): 26.6566757493188,
 ('Friday', 9): 31.746630727762803,
 ('Friday', 10): 34.93766937669377,
 ('Friday', 11): 36.696883852691215,
 ('Friday', 12): 38.91812865497076,
 ('Friday', 13): 36.96470588235294,
 ('Friday', 14): 37.78285714285714,
 ('Friday', 15): 41.94767441860465,
 ('Friday', 16): 48.07848837209303,
 ('Friday', 17): 54.46511627906977,
 ('Friday', 18): 50.388739946380696,
 ('Friday', 19): 44.632876712328766,
 ('Friday', 20): 38.083109919571044,
 ('Friday', 21): 36.524324324324326,
 ('Friday', 22): 33.56318681318681,
 ('Friday', 23): 6.392953929539296,
 ('Monday', 0): 16.944881889763778,
 ('Monday', 1): 2.5701754385964914,
 ('Monday', 2): 0.17592592592592593,
 ('Monday', 3): 0.0930232558139534

In [8]:
def is_raining(rain_mm):
    if rain_mm > 0:
        return True
    else:
        return False

def get_weather_forecast(target_date):
    cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    openmeteo = openmeteo_requests.Client(session = retry_session)

    # Make sure all required weather variables are listed here
    # The order of variables in hourly or daily is important to assign them correctly below
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": 37.8716,
        "longitude": -122.2728,
        "hourly": ["temperature_2m", "rain", "precipitation_probability"],
        "temperature_unit": "fahrenheit",
        "timezone": "America/Los_Angeles",
        "start_date": target_date,
        "end_date": target_date
    }
    responses = openmeteo.weather_api(url, params=params)

    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]
    """
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
    """
    
    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_rain = hourly.Variables(1).ValuesAsNumpy()
    hourly_precipitation_probability = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {"date": pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
        end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
        freq = pd.Timedelta(seconds = hourly.Interval()),
        inclusive = "left"
    )}

    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["rain"] = hourly_rain
    hourly_data["precipitation_probability"] = hourly_precipitation_probability

    hourly_dataframe = pd.DataFrame(data = hourly_data)
    hourly_dataframe['date'] = hourly_dataframe['date'].dt.tz_convert('America/Los_Angeles')

    hourly_dataframe.rename(columns={'rain': 'rain_mm'}, inplace=True)
    hourly_dataframe["hour"] = hourly_dataframe["date"].dt.hour
    hourly_dataframe["is_raining"] = hourly_dataframe["rain_mm"].apply(is_raining)

    hourly_dataframe.drop(columns=['date','temperature_2m'], inplace=True)

    return hourly_dataframe


In [17]:
def predict_gym_traffic(target_date, target_hour):
    # Get if it will rain
    forecast = get_weather_forecast(target_date)
    is_raining = forecast[forecast["hour"] == target_hour]["is_raining"].values[0]

    # Get if date is a holiday
    if target_date in ca_holidays:
        is_holiday = True
    else:
        is_holiday = False

    # Get day of the week
    target_date = pd.to_datetime(target_date)
    day = target_date.day_name()
    key = (day, target_hour)

    rainpenalty = rain_penalty[rain_penalty["day_of_week"] == day]["rain_penalty"].values[0]
    forecast_base = reference_base.get(key, 0)

    forecast_final = forecast_base

    if is_raining:
        forecast_final += rainpenalty
    if is_holiday:
        forecast_final += holiday_penalty
    
    return max(0, round(forecast_final))

# predict_gym_traffic('2026-02-05',17) 